In [ ]:
import pandas as pd
import yaml
import re
import os
DB_FOLDER = f"dataset\database_v3"

In [66]:
trans_df = pd.read_csv(os.path.join(DB_FOLDER, "transaction.csv"))
trans_df

,transaction_id,timestep,Lease Commencement Date,project_id,Project Name,No of Bedroom,Floor Area (SQFT),sqft,Monthly Rent ($),rent_per_sqft
0,1,0,1999-11-01,844,FAR EAST PLAZA,NaN,500 to 600,550.0,1690,3.072727
1,2,0,1999-11-01,2803,THE PLAZA,NaN,700 to 800,750.0,2648,3.530667
2,3,0,1999-11-01,2026,REGENT PARK,NaN,"1,100 to 1,200",1150.0,2500,2.173913
3,4,0,1999-11-01,1529,MANDARIN GARDENS,NaN,800 to 900,850.0,2300,2.705882
4,5,0,1999-11-01,1529,MANDARIN GARDENS,NaN,800 to 900,850.0,2166,2.548235
...,...,...,...,...,...,...,...,...,...,...
1294499,1294500,288,2026-01-01,614,D'LEEDON,1.0,600 to 700,650.0,3500,5.384615
1294500,1294501,288,2026-01-01,614,D'LEEDON,3.0,"1,200 to 1,300",1250.0,7200,5.760000
1294501,1294502,288,2026-01-01,614,D'LEEDON,3.0,"1,500 to 1,600",1550.0,7200,4.645161
1294502,1294503,288,2026-01-01,614,D'LEEDON,2.0,"1,000 to 1,100",1050.0,5100,4.857143


In [67]:
trans_df['No of Bedroom'] = trans_df['No of Bedroom'].fillna(0)
majority_map = (
    trans_df.loc[trans_df['No of Bedroom'] != 0]
    .groupby(['project_id', 'sqft'])['No of Bedroom']
    .agg(lambda x: x.mode().iloc[0])
)

mask = trans_df['No of Bedroom'] == 0

trans_df.loc[mask, 'No of Bedroom'] = trans_df.loc[mask, ['project_id', 'sqft']] \
    .apply(lambda row: majority_map.get((row['project_id'], row['sqft']), 0), axis=1)

In [68]:
grouped_df = (
    trans_df.groupby(['project_id', 'sqft', 'No of Bedroom'])
    .size()
    .reset_index(name='count')
)
grouped_df['No of Bedroom'] = grouped_df['No of Bedroom'].astype('Int64')
# assign node_id
grouped_df = grouped_df.sort_values(['project_id', 'sqft', 'No of Bedroom']).reset_index(drop=True)
grouped_df['node_id'] = grouped_df.index
cols_order = ['node_id', 'project_id', 'sqft', 'No of Bedroom']
grouped_df = grouped_df[cols_order]
grouped_df.to_csv(os.path.join(DB_FOLDER, "node_id.csv"), index=False)

# --------------------------------------------------
# 1. One-hot encode bedroom (ignore 0)
# --------------------------------------------------
grouped_df['No of Bedroom'] = pd.to_numeric(grouped_df['No of Bedroom'], errors='coerce')

# keep only valid bedroom 1–8 for one-hot
bedroom_df = grouped_df.copy()

# create one-hot
bedroom_df = pd.get_dummies(
    bedroom_df,
    columns=['No of Bedroom'],
    prefix='bedroom',
    dtype='int8'
)

# remove bedroom_0 if exists
bedroom_cols = [c for c in bedroom_df.columns if c.startswith('bedroom_') and not c.endswith('_0')]

# --------------------------------------------------
# 2. Normalize sqft to [-1, 1]
# --------------------------------------------------
sqft_min = bedroom_df['sqft'].min()
sqft_max = bedroom_df['sqft'].max()
sqft_range = sqft_max - sqft_min if (sqft_max - sqft_min) != 0 else 1

bedroom_df['sqft'] = 2 * (bedroom_df['sqft'] - sqft_min) / sqft_range - 1
bedroom_df['sqft'] = bedroom_df['sqft'].fillna(0)

# --------------------------------------------------
# 3. Save processed node file
# --------------------------------------------------
node_cols = ['node_id', 'project_id', 'sqft'] + bedroom_cols
node_df = bedroom_df[node_cols].sort_values('node_id').reset_index(drop=True)

node_df.to_csv(os.path.join(DB_FOLDER, "node_processed.csv"), index=False)

# --------------------------------------------------
# 4. Append to config.yaml
# --------------------------------------------------
config_path = os.path.join(DB_FOLDER, "config.yaml")

if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f) or {}
else:
    config = {}

config["node_processing"] = {
    "bedroom_onehot_cols": bedroom_cols,
    "sqft_normalization": {
        "method": "minmax_-1_1",
        "min": float(sqft_min),
        "max": float(sqft_max)
    },
    "bedroom_range": [1, 8],
    "ignore_bedroom_value": 0
}

with open(config_path, "w") as f:
    yaml.dump(config, f, sort_keys=False)

In [69]:
trans_df = trans_df.merge(
    grouped_df[['project_id', 'sqft', 'No of Bedroom', 'node_id']],
    on=['project_id', 'sqft', 'No of Bedroom'],
    how='left'
)
trans_df

,transaction_id,timestep,Lease Commencement Date,project_id,Project Name,No of Bedroom,Floor Area (SQFT),sqft,Monthly Rent ($),rent_per_sqft,node_id
0,1,0,1999-11-01,844,FAR EAST PLAZA,0.0,500 to 600,550.0,1690,3.072727,5778
1,2,0,1999-11-01,2803,THE PLAZA,2.0,700 to 800,750.0,2648,3.530667,19663
2,3,0,1999-11-01,2026,REGENT PARK,3.0,"1,100 to 1,200",1150.0,2500,2.173913,13598
3,4,0,1999-11-01,1529,MANDARIN GARDENS,2.0,800 to 900,850.0,2300,2.705882,10005
4,5,0,1999-11-01,1529,MANDARIN GARDENS,2.0,800 to 900,850.0,2166,2.548235,10005
...,...,...,...,...,...,...,...,...,...,...,...
1294499,1294500,288,2026-01-01,614,D'LEEDON,1.0,600 to 700,650.0,3500,5.384615,4231
1294500,1294501,288,2026-01-01,614,D'LEEDON,3.0,"1,200 to 1,300",1250.0,7200,5.760000,4243
1294501,1294502,288,2026-01-01,614,D'LEEDON,3.0,"1,500 to 1,600",1550.0,7200,4.645161,4246
1294502,1294503,288,2026-01-01,614,D'LEEDON,2.0,"1,000 to 1,100",1050.0,5100,4.857143,4238


In [70]:
import os
import pandas as pd

output_dir = os.path.join(DB_FOLDER, "timestep")
os.makedirs(output_dir, exist_ok=True)

# ensure date is datetime
trans_df['Lease Commencement Date'] = pd.to_datetime(
    trans_df['Lease Commencement Date'], errors='coerce'
)

# rename for consistency
trans_df = trans_df.rename(columns={
    'No of Bedroom': 'no_of_room'
})

# columns to keep
cols = [
    'timestep', 'node_id', 'project_id',
    'no_of_room', 'sqft',
    'rent_per_sqft', 'Lease Commencement Date'
]

df_use = trans_df[cols].copy()

# --------------------------------------------------
# 1. aggregate observed rows first
# --------------------------------------------------
agg_df = (
    df_use.groupby(
        ['timestep', 'node_id', 'project_id', 'no_of_room', 'sqft'],
        as_index=False
    )
    .agg({
        'rent_per_sqft': 'mean',
        'Lease Commencement Date': 'first'
    })
)

# observed rows
agg_df['y_mask'] = 1

# --------------------------------------------------
# 2. interpolate missing timesteps within each node
# --------------------------------------------------
filled_list = []

for node_id, group in agg_df.groupby('node_id'):
    group = group.sort_values('timestep').copy()

    # static info for this node
    project_id = group['project_id'].iloc[0]
    no_of_room = group['no_of_room'].iloc[0]
    sqft = group['sqft'].iloc[0]

    # build full timestep range only between first and last observed timestep
    full_timestep = pd.DataFrame({
        'timestep': range(
            int(group['timestep'].min()),
            int(group['timestep'].max()) + 1
        )
    })

    # merge to create missing intermediate rows
    node_df = full_timestep.merge(
        group,
        on='timestep',
        how='left'
    )

    # fill static columns
    node_df['node_id'] = node_id
    node_df['project_id'] = node_df['project_id'].fillna(project_id)
    node_df['no_of_room'] = node_df['no_of_room'].fillna(no_of_room)
    node_df['sqft'] = node_df['sqft'].fillna(sqft)

    # mark inserted rows
    node_df['y_mask'] = node_df['y_mask'].fillna(0).astype(int)

    # interpolate rent_per_sqft only inside observed range
    node_df['rent_per_sqft'] = node_df['rent_per_sqft'].interpolate(method='linear')

    # handle Lease Commencement Date by timestep mapping later, so drop for now
    filled_list.append(node_df)

filled_df = pd.concat(filled_list, ignore_index=True)

# --------------------------------------------------
# 3. restore Lease Commencement Date from timestep mapping
# --------------------------------------------------
timestep_date_map = (
    agg_df[['timestep', 'Lease Commencement Date']]
    .drop_duplicates(subset=['timestep'])
    .sort_values('timestep')
)

filled_df = filled_df.drop(columns=['Lease Commencement Date'], errors='ignore')
filled_df = filled_df.merge(
    timestep_date_map,
    on='timestep',
    how='left'
)

filled_df = filled_df.sort_values(['timestep', 'node_id']).reset_index(drop=True)

# --------------------------------------------------
# 4. split by timestep
# --------------------------------------------------
for t, group in filled_df.groupby('timestep'):
    group = group.sort_values('node_id')

    date = group['Lease Commencement Date'].iloc[0]
    date_str = date.strftime('%Y-%m') if pd.notna(date) else 'unknown'

    file_name = f"{int(t)}_{date_str}.csv"
    file_path = os.path.join(output_dir, file_name)

    group_to_save = group[
        ['timestep', 'node_id', 'project_id', 'no_of_room', 'sqft', 'rent_per_sqft', 'y_mask']
    ]

    group_to_save.to_csv(file_path, index=False)